In [1]:
import pandas as pd
import random


def generate_client_ids(n=10):
    """Generate n random 10-digit client ID strings."""
    ids = []
    for _ in range(n):
        # Ensure it's exactly 10 digits (no leading digit loss)
        id_code = "".join(str(random.randint(0, 9)) for _ in range(10))
        ids.append(id_code)
    return ids


# Number of entries (default = 10)
n = 10

# Create DataFrame
df = pd.DataFrame({"client_idcode": generate_client_ids(n)})

# Save to CSV
df.to_csv("treatment_docs.csv", index=False)

print("CSV file 'treatment_docs.csv' created!")
print(df.head())

# Replace this with your cohort (Hospital numbers)

CSV file 'treatment_docs.csv' created!
  client_idcode
0    2586065735
1    1769056510
2    7547810497
3    5932401396
4    3800333100


In [2]:
import numpy as np
import os
import sys
import shutil

# Fix the random seed for reproducibility in unit testing

random_seed_value = 42

np.random.seed(random_seed_value)

random.seed(random_seed_value)

In [3]:
# 1. Print the current working directory
print("Current Working Directory:", os.getcwd())

# 2. Print Python's sys.path
print("Python Path:", sys.path)

Current Working Directory: /workspaces/pat2vec/notebooks
Python Path: ['/usr/local/lib/python310.zip', '/usr/local/lib/python3.10', '/usr/local/lib/python3.10/lib-dynload', '', '/home/vscode/.local/lib/python3.10/site-packages', '__editable__.pat2vec-0.3.4.finder.__path_hook__', '/usr/local/lib/python3.10/site-packages']


In [4]:
# remove dir
clear_previous_outputs = True

if clear_previous_outputs:
    shutil.rmtree("new_project", ignore_errors=True)

    shutil.rmtree("new_project_ipw", ignore_errors=True)

    shutil.rmtree("treatment_doc_extract", ignore_errors=True)

In [5]:
# Ensure dependencies are on path

# Get the current working directory
current_dir = os.getcwd()

# Define relative paths from the current working directory
path_to_medcat_model_pack = os.path.abspath(
    os.path.join(
        current_dir,
        "..",
        "..",
        "medcat_models",
        "medcat_model_pack_422d1d38fc58f158.zip",
    )
)

path_to_snomed_ct_file = os.path.abspath(
    os.path.join(
        current_dir,
        "..",
        "..",
        "snomed",
        "SnomedCT_InternationalRF2_PRODUCTION_20231101T120000Z",
        "SnomedCT_InternationalRF2_PRODUCTION_20231101T120000Z",
        "Full",
        "Terminology",
        "sct2_StatedRelationship_Full_INT_20231101.txt",
    )
)

# Define the relative path
path_to_gloabl_files = "../../"

additional_path_to_pat2vec = "pat2vec"

additional_path_to_pat2vec = os.path.abspath(
    os.path.join(path_to_gloabl_files, additional_path_to_pat2vec)
)

# Get the absolute path of the current working directory
current_dir = os.getcwd()

# Combine the current directory with the relative path
absolute_path = os.path.abspath(os.path.join(current_dir, path_to_gloabl_files))

# Usage examples
print(path_to_medcat_model_pack)
print(path_to_snomed_ct_file)
print(path_to_gloabl_files)
print(additional_path_to_pat2vec)

/workspaces/medcat_models/medcat_model_pack_422d1d38fc58f158.zip
/workspaces/snomed/SnomedCT_InternationalRF2_PRODUCTION_20231101T120000Z/SnomedCT_InternationalRF2_PRODUCTION_20231101T120000Z/Full/Terminology/sct2_StatedRelationship_Full_INT_20231101.txt
../../
/workspaces/pat2vec


In [6]:
sys.path.insert(0, path_to_gloabl_files)
sys.path.insert(0, additional_path_to_pat2vec)

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

# Add the grandparent directory of the current directory to the Python path
grandparent_dir = os.path.dirname(parent_dir)
sys.path.append(grandparent_dir)

### Set up logger

In [7]:
from pat2vec.util.logger_setup import setup_logger

# Get the logger
logger = setup_logger()

/home/vscode/.local/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


INFO: PyTorch version 2.8.0 available.


INFO: Polars version 1.42.0 available.


### Database Backend Setup

The `pat2vec` pipeline now supports a database backend for handling large datasets and ensuring data persistence.

For this example, we will set up an **ephemeral SQLite database**. This allows us to run the pipeline with a clean state, storing all intermediate outputs (raw data, annotations, features) in a local file that is reset on each run.

The following code block:
1.  Defines the database path within the project's output directory.
2.  Cleans up any existing database file to prevent data conflicts during testing.
3.  Generates the SQLAlchemy connection string required by the `config_class`.


In [8]:
import os

# --- Ephemeral Database Setup ---
# Define a name and path for your temporary database file.
# Placing it in the project's output directory is a good practice.
PROJ_NAME = "new_project"
DB_FILENAME = "temp_test_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Teardown: Ensure the old database is deleted before each run for a clean state.
try:
    os.remove(DB_PATH)
    print(f"Removed old database file: {DB_PATH}")
except FileNotFoundError:
    print(f"No old database file to remove. A new one will be created at: {DB_PATH}")

# Create the SQLAlchemy connection string for SQLite.
# The 'sqlite:///' prefix indicates a file-based database.
db_connection_string = f"sqlite:///{DB_PATH}"

print(f"Database connection string set to: {db_connection_string}")

No old database file to remove. A new one will be created at: new_project/outputs/temp_test_db.sqlite
Database connection string set to: sqlite:///new_project/outputs/temp_test_db.sqlite


In [9]:
from pat2vec.util.config_pat2vec import config_class
from datetime import datetime
from tqdm import tqdm
from pat2vec.util.post_processing import extract_datetime_to_column
from dateutil.relativedelta import relativedelta
import pandas as pd
from typing import Dict, List, Optional, Union

# Configuration dictionary for main options in pat2vec
main_options_dict = {
    "demo": True,  # Enable demographic information (Ethnicity mapped to UK census categories, age, death).
    "bmi": True,  # Enable BMI (Body Mass Index) information.
    "bloods": True,  # Enable blood-related information
    "drugs": True,  # Enable drug-related information
    "diagnostics": True,  # Enable diagnostic information
    "core_02": True,  # Enable core_02 information
    "bed": True,  # Enable bed n information
    "vte_status": True,  # Enable VTE () status information
    "hosp_site": True,  # Enable hospital site information
    "core_resus": True,  # Enable core resuscitation information
    "news": True,  # Enable NEWS (National Early Warning Score) information
    "smoking": True,  # Enable smoking-related information
    "annotations": True,  # Enable EPR documents annotations via MedCat
    "annotations_mrc": True,  # Enable MRC (Additional clinical note observations index) annotations via MedCat
    "negated_presence_annotations": False,  # Enable or disable negated presence annotations
    "appointments": False,  # Enable appointments information
    "annotations_reports": False,  # Enable reports information
    "textual_obs": False,  # Enable textual observations (basic_observations index) annotations via MedCat
    "covid": True,  # Enable covid test results
    "epic_encounters": False,  # Enable epic_encounters data source
    "epic_clinical_notes": True,  # Enable epic_clinical_notes data source
    "epic_medical_history": True,  # Enable epic_medical_history data source
    "epic_orders": True,  # Enable epic_orders data source
    "epic_orders_annotations": True,  # Enable epic_orders_annotations data source
    "epic_lab_results": True,  # Enable epic_lab_results data source
    "epic_patients": True,  # Enable epic_patients data source
    "epic_imaging_reports": True,  # Enable epic_imaging_reports data source
    "epic_clinical_notes_appointments": True,  # Enable epic_clinical_notes_appointments data source
}

# Configuration dictionary for annotation filtering, only base annotations meeting this threshold will be included.
annot_filter_arguments = {
    "acc": 0.8,  # base concept accuracy
    "types": [
        "qualifier value",
        "procedure",
        "substance",
        "finding",
        "environment",
        "disorder",
        "observable entity",
    ],  # umls list of types for medcat filter
    # 'types': ['qualifier value', 'procedure', 'substance', 'finding', 'environment', 'disorder', 'observable entity', 'organism', 'phenomenon', 'anatomy', 'conceptual entity', 'physical object', 'intellectual product', 'occupation or discipline', 'mental or behavioral dysfunction', 'geographic area', 'population group', 'biomedical or dental material', 'medical device', 'classification', 'regulation or law', 'health care activity', 'health care related organization', 'professional or occupational group', 'group', 'attribute', 'individual behavior']
    "Time_Value": [
        "Recent",
        "Past",
    ],  # Specify the values you want to include in a list. Must be defined in medcat model. # Example ['Recent', 'Past', 'Subject/Experiencer']
    "Time_Confidence": 0.8,  # Specify the confidence threshold as a float
    "Presence_Value": ["True"],  # Specify the values you want to include in a list
    "Presence_Confidence": 0.8,  # Specify the confidence threshold as a float
    "Subject_Value": ["Patient"],  # Specify the values you want to include in a list
    "Subject_Confidence": 0.8,  # Specify the confidence threshold as a float
}

# Filter data batches by terms before processing.

epr_docs_term_regex: Optional[Union[str, None]] = None
mct_docs_term_regex: Optional[Union[str, None]] = None

# Example bloods_filter_term_list: Optional[Union[List[str], None]] = ['wbc'] # This will only include basic observations with this item name analysed.
bloods_filter_term_list: Optional[Union[List[str], None]] = None

# Example mct_docs_document_type_filter_list: Optional[Union[List[str], None]] = ['KHMDC Integrated report'] # This will only include documents with this document type field value.

mct_docs_document_type_filter_list: Optional[Union[List[str], None]] = None
epr_docs_document_type_filter_list: Optional[Union[List[str], None]] = None

data_type_filter_dict: Dict[str, any] = {
    "filter_term_lists": {
        "epr_docs": epr_docs_document_type_filter_list,
        "mct_docs": mct_docs_document_type_filter_list,
        "bloods": bloods_filter_term_list,
    },
    "epr_docs_term_regex": epr_docs_term_regex,
    "mct_docs_term_regex": mct_docs_term_regex,
}

# Example date settings:
# start_date=(datetime(2020, 1, 1)) Start date for processing

# Define the length of the time window, example 1 year and 15 days, only data within this window will be processed.
# years=1,      # Number of years to add to the start date
# months=0,  # Number of months to add to the start date
# days=15,  # Number of days to add to the start date

# Define the interval between time windows. Example 1 year. Each vector/row output will be based on this interval.
# time_window_interval_delta = relativedelta(years=1)

# lookback = True #This determines the direction of the time length window. True = backward, False = forward. Our time window (+1 years, 15 days) is therefore 2020, 1, 1 - 2021, 1, 15.

# IPW settings:

# Init config obj

# Hypothetical date config_obj configuration:
# I want all patients data between Feb 2015 and Jul 2020. This date window will extract and create the batched patient data for this time window.

# global_start_year=2015,
# global_start_month=2,
# global_end_year=2020,
# global_end_month=6,
# global_start_day = 1,
# global_end_day = 1,

# I want patient vectors starting from Feb 2019 to Feb 2020 as I would like to see if X medical event is recorded on those taking medication Y
# start_date=(datetime(2019, 2, 1)),
# years=1,
# months=0,
# days=0,
# lookback = False # 2019 to 2020 is forward in time.
# I would like a single vector for each patient
# time_window_interval_delta = relativedelta(years=1)
# I would like 1 vector per month per patient for the 1 year time window
# time_window_interval_delta = relativedelta(months=1)

# Creating a configuration object for a specific task or project
config_obj = config_class(
    remote_dump=False,  # Flag for remote data dumping. partially deprecated.
    suffix="",  # Suffix for file names
    treatment_doc_filename="test_files/treatment_docs.csv",  # Filename for treatment documentation
    treatment_control_ratio_n=1,  # Ratio for treatment to control
    proj_name="new_project",  # Project name. patient data batches and vectors stored here.
    current_path_dir="",  # Current path directory
    main_options=main_options_dict,  # Dictionary for main options
    start_date=(datetime(1995, 1, 1)),  # Starting date for processing
    years=30,  # Number of years to add to the start date. Set the duration of the time window. Window is defined as the start date + years/months/days set here.
    months=0,  # Number of months to add to the start date
    days=0,  # Number of days to add to the start date
    batch_mode=True,  # Flag for batch processing mode. Only functioning mode.
    store_annot=True,  # Flag to store annotations. partially deprecated.
    share_sftp=True,  # Flag for sharing via SFTP. partially deprecated
    multi_process=False,  # Flag for multi-process execution. deprecated.
    strip_list=True,  # Flag for stripping lists, this will check for completed patients before starting to avoid redundancy.
    verbosity=0,  # Verbosity level 0-9 printing debug messages
    random_seed_val=random_seed_value,  # Random seed value for reproducibility of controls.
    testing=True,  # Flag for testing mode. Will use dummy data.
    dummy_medcat_model=True,  # Flag for dummy MedCAT model, used if testing == True, this will simulate a MedCAT model.
    use_controls=False,  # If true this will add desired ratio of controls at random from global pool, requires configuring with a master list of patients.
    medcat=False,  # Flag for MedCAT processing. #will load medcat into memory and use for annotating.
    start_time=datetime.now(),  # Current timestamp as the start time for logging and progress bar
    patient_id_column_name="auto",  # Column name for patient ID, auto will try to find it. Example "client_idcode"
    annot_filter_options=annot_filter_arguments,  # Annotation filtering options
    # Global start year. #set the limits of the time window data can be drawn from. Start should not precede start date set above.
    global_start_year=1995,  # Global dates are overwritten by individual patient windows to match patient window. # Ensure that global start year/month/day is before end year/month/day
    global_start_month=1,  # Global start month
    global_end_year=2025,  # Global end year
    global_end_month=1,  # Global end month
    global_start_day=1,
    global_end_day=1,
    ## Use these if each patient has their own individual time window. Requires preparing a table of start dates.
    # individual_patient_window = True,
    # individual_patient_window_df = pd.read_csv('ipw_overlap.csv'),
    # individual_patient_window_start_column_name = 'updatetime_manual_offset',
    # individual_patient_id_column_name = 'client_idcode',
    # individual_patient_window_controls_method = 'full',
    shuffle_pat_list=False,  # Flag for shuffling patient list
    time_window_interval_delta=relativedelta(
        years=31
    ),  # specify the time window to collapse each feature vector into, years=1 is one vector per year within the global time window
    split_clinical_notes=True,  # will split clinical notes by date and treat as individual documents with extracted dates. Requires note splitter module.
    lookback=False,  # when calculating individual patient window from table of start dates, will calculate backwards in time if true. Else Forwards. When calculating from global start date, will calculate backwards or forwards respectively.
    add_icd10=False,  # append icd 10 codes to annot batches. Can be found under current_pat_documents_annotations/%client_idcode%.csv.
    add_opc4s=False,  # needs icd10 true also. Can be found under current_pat_documents_annotations/%client_idcode%.csv
    override_medcat_model_path=path_to_medcat_model_pack,  # Force medcat model path, if None uses defaults for env. #Can be set in paths.py with medcat_path = %path to medcat model pack.zip"
    data_type_filter_dict=None,  # Dictionary for data type filter, see examples above.
    filter_split_notes=True,  # If enabled, will reapply global time window filter post clinical note splitting. Recommended to enable if split notes enabled.
    prefetch_pat_batches=False,  # If enabled, will fetch batches for entire patient list and pre poulate batch folders with individual pat batches. Out of memory issues.
    sample_treatment_docs=5,  # If int > 0, will sample treatment documents from the treatment_docs.csv file. This is useful for testing and debugging / pilot run purposes.
    credentials_path="../util/credentials.py",
    storage_backend="database",
    db_connection_string=db_connection_string,
)

2026-08-05 20:45:23,955 - pat2vec.util.config_pat2vec - INFO - credentials_path: ../util/credentials.py


2026-08-05 20:45:23,955 - pat2vec.util.config_pat2vec - WARNING - Warning: client_idcode_term_name 'client_idcode.keyword' is not case inclusive.


2026-08-05 20:45:23,956 - pat2vec.util.current_pat_batch_path_methods - INFO - Created and verified the following paths:


2026-08-05 20:45:23,956 - pat2vec.util.current_pat_batch_path_methods - INFO - /workspaces/pat2vec/notebooks/new_project/current_pat_annots_mrc_parts/


2026-08-05 20:45:23,956 - pat2vec.util.current_pat_batch_path_methods - INFO - /workspaces/pat2vec/notebooks/new_project/current_pat_annots_parts/


...

2026-08-05 20:45:23,964 - pat2vec.util.config_pat2vec - INFO - Expected time interval vectors per patient: 0


2026-08-05 20:45:23,964 - pat2vec.util.config_pat2vec - INFO - Time interval vectors will span the following dates: 1995-01-01 00:00:00 to 2025-01-01 00:00:00


2026-08-05 20:45:23,964 - pat2vec.util.config_pat2vec - INFO - Setting test options


2026-08-05 20:45:23,965 - pat2vec.util.config_pat2vec - INFO - Defaulting test_data_path to: test_files/treatment_docs.csv


2026-08-05 20:45:23,965 - pat2vec.util.config_pat2vec - INFO - Updating main options with implemented test options


In [10]:
from pat2vec.main_pat2vec import main

In [11]:
pat2vec_obj = main(
    cogstack=True,
    use_filter=False,
    json_filter_path=None,
    random_seed_val=42,
    hostname=None,
    config_obj=config_obj,
)

INFO: Initialized cohort_searcher_with_terms_and_search_dummy function.


2026-08-05 20:45:23,974 - pat2vec.util.get_best_gpu - INFO - Setting NO GPU, most free memory: 3999 MB!


Sample invalid codes: ['P0IFD0TV', 'VF8MDD4V', 'V945NQ4F', 'V945NQ4F', 'VC58DRC1', 'P64CIYHE']
failed to analyze_client_codes
empty vocabulary; perhaps the documents only contain stop words
Sampling 5 of 6 available treatment docs.
all_patient_list size now: 5


Bar desc:   0%|          | 0/5 [00:00<?, ?it/s]

2026-08-05 20:45:23,976 - pat2vec.util.methods_get_medcat - INFO - Returning dummy_CAT for testing.


INFO: No pre-existing filters found in model. Processing all entities.


View patient list

In [12]:
pat2vec_obj.all_patient_list[0:8]

['P0IFD0TV', 'VF8MDD4V', 'P64CIYHE', 'V945NQ4F', 'VC58DRC1']

In [13]:
pat2vec_obj.config_obj.date_list

[(1995, 1, 1)]

Make pat vectors for pat 0

In [14]:
pat2vec_obj.pat_maker(0)

s: 0 | P0IFD0TV | task: Pat_maker called on 0... | {}:   0%|          | 0/5 [00:00<?, ?it/s]

s: 0 | P0IFD0TV | task: Pat_maker called on 0... | {}:   0%|          | 0/5 [00:00<?, ?it/s]

2026-08-05 20:45:23,991 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_epr_docs: Table 'raw_data_raw_epr_docs' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:23,992 - pat2vec.util.get_dummy_data_cohort_searcher - INFO - Generating 1 dummy EPR docs for 1 patients, e.g., P0IFD0TV


2026-08-05 20:45:24,022 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_mct_docs: Table 'raw_data_raw_mct_docs' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


...

s: 0 | P0IFD0TV_(1995, 1, 1) | task: Columns n=115 | {}:   0%|          | 0/5 [00:00<?, ?it/s]

s: 0 | P0IFD0TV_(1995, 1, 1) | task: Columns n=115 | {}:   0%|          | 0/5 [00:00<?, ?it/s]

/workspaces/pat2vec/pat2vec/util/helper_functions.py:251: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features_df[id_column] = patient_id
INFO: Using JSON packing mode to avoid column limit issues (116 columns)


INFO: Packing 116 features into JSON for patient P0IFD0TV


WARN: No data available for patient P0IFD0TV in epic_orders annotations

=== DEBUG _get_patient_data_batches END ===
DEBUG helper_functions: Before drop - columns: ['client_idcode', 'document_guid', 'document_description', 'body_analysed', 'updatetime', 'clientvisit_visitidcode', 'search_term']
DEBUG helper_functions: Dropping column search_term
DEBUG helper_functions: After drop - columns: ['client_idcode', 'document_guid', 'document_description', 'body_analysed', 'updatetime', 'clientvisit_visitidcode']
DEBUG helper_functions: Before drop - columns: ['observation_guid', 'client_idcode', 'obscatalogmasteritem_displayname', 'observation_valuetext_analysed', 'observationdocument_recordeddtm', 'clientvisit_visitidcode', 'updatetime', 'source_file']
DEBUG helper_functions: After drop - columns: ['observation_guid', 'client_idcode', 'obscatalogmasteritem_displayname', 'observation_valuetext_analysed', 'observationdocument_recordeddtm', 'clientvisit_visitidcode', 'updatetime', 'source_file'

In [15]:
# Remove specific patient raw documents and annotations:
from pat2vec.util.post_processing import remove_file_from_paths

# remove_file_from_paths(pat2vec_obj.all_patient_list[i])

In [16]:
# Define the maximum number of retries
MAX_RETRIES = 3

# Iterate through the patient list starting from index 0
for i in tqdm(range(0, len(pat2vec_obj.all_patient_list))):
    retries = 0
    success = False

    while retries < MAX_RETRIES and not success:
        try:
            # Try to process the patient
            pat2vec_obj.pat_maker(i)
            success = True  # Mark as successful if no exception is raised

        except KeyError as e:
            # Handle specific exception
            print(f"KeyError at index {i}: {e}. Retrying after removal...")
            remove_file_from_paths(pat2vec_obj.all_patient_list[i])
            retries += 1

        except Exception as e:
            # Handle generic exceptions
            print(f"Exception at index {i}: {e}. Skipping this patient...")
            break  # Break the retry loop for non-retryable exceptions

        finally:
            pat2vec_obj.t.update(1)  # Update progress

    if not success:
        print(f"Failed to process index {i} after {MAX_RETRIES} retries.")

pat2vec_obj.t.close()

  0%|          | 0/5 [00:00<?, ?it/s]

s: 0 | P0IFD0TV | task: Pat_maker called on 0... | {}:   0%|          | 0/5 [00:00<?, ?it/s]  

s: 0 | P0IFD0TV | task: Pat_maker called on 0... | {}:   0%|          | 0/5 [00:00<?, ?it/s]


=== DEBUG _get_patient_data_batches START ===
Patient: P0IFD0TV
DEBUG: About to fetch 22 standard batches


2026-08-05 20:45:24,433 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for annotations.ann_epic_orders: Table 'annotations_ann_epic_orders' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


...

INFO: Packing 109 features into JSON for patient VC58DRC1


s: 0 | VC58DRC1_(1995, 1, 1) | task: Columns n=108 | {}: 100%|██████████| 5/5 [00:02<00:00,  2.51it/s]

100%|██████████| 5/5 [00:01<00:00,  2.77it/s]

100%|██████████| 5/5 [00:01<00:00,  3.07it/s]


s: 0 | VC58DRC1_(1995, 1, 1) | task: Columns n=108 | {}: 100%|██████████| 5/5 [00:02<00:00,  2.43it/s]

In [17]:
from pat2vec.util.helper_functions import get_all_features
import os

# Retrieve all features directly from the database
# This replaces the process_csv_files step
df = get_all_features(pat2vec_obj.config_obj)

# Define output path (matching your previous workflow)
output_directory = f"{pat2vec_obj.proj_name}/output_directory"
output_file = f"{output_directory}/output_file.csv"

# Create directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Save the aggregated features to a single CSV file
df.to_csv(output_file, index=False)

print(f"Features exported to {output_file}")
print(f"Total shape: {df.shape}")

INFO: get_all_features called with backend: database


Features exported to new_project/output_directory/output_file.csv
Total shape: (5, 305)


In [18]:
df = pd.read_csv(output_file)

In [19]:
df = extract_datetime_to_column(df)

In [20]:
df

,93pct,95pct,96pct,98pct,Breast Cancer Diagnosis_days-since-last-diagnostic-order,Breast Cancer Diagnosis_num-diagnostic-order,Breast Cancer Staging_days-since-last-diagnostic-order,Breast Cancer Staging_num-diagnostic-order,Breast Cancer Survivorship Counseling_days-since-last-diagnostic-order,Breast Cancer Survivorship Counseling_num-diagnostic-order,...,weight_max,weight_mean,weight_median,weight_min,weight_std,whether_days-since-last-drug-order,whether_num-drug-order,wish_days-since-last-diagnostic-order,wish_num-diagnostic-order,extracted_datetime_stamp
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9509.0,1.0,...,109.15,109.15,109.15,109.15,0.0,NaN,NaN,NaN,NaN,1995-01-01
1,1.0,1.0,NaN,1.0,5928.0,1.0,8459.0,1.0,NaN,NaN,...,51.71,51.71,51.71,51.71,0.0,0.0,1.0,0.0,1.0,1995-01-01
2,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,148.85,148.85,148.85,148.85,0.0,NaN,NaN,NaN,NaN,1995-01-01
3,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,148.85,148.85,148.85,148.85,0.0,NaN,NaN,NaN,NaN,1995-01-01
4,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1995-01-01


#### Build all document batches dataframe:

In [21]:
# This will merge all document source batches into a single file. This is useful for filtering. May produce a large file.

from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

all_pat_list = pat2vec_obj.all_patient_list

dfd = build_merged_epr_mct_doc_df(all_pat_list, pat2vec_obj.config_obj, overwrite=True)

# dfd = pd.read_csv(dfd)

2026-08-05 20:45:26,334 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: docs_mct_epr.csv. Processing 5 patient(s) with chunk_size=1.


Merging docs_mct_epr.csv:   0%|          | 0/5 [00:00<?, ?chunk/s]

2026-08-05 20:45:27,142 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_textual_obs: Table 'raw_data_raw_textual_obs' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:27,144 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_reports: Table 'raw_data_raw_reports' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:27,145 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_epic_medical_history: Table 'raw_data_raw_epic_medical_history' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


...

2026-08-05 20:45:33,822 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_epic_imaging_reports: Table 'raw_data_raw_epic_imaging_reports' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:33,823 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for raw_data.raw_epic_orders: Table 'raw_data_raw_epic_orders' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


Merging docs_mct_epr.csv:  40%|████      | 2/5 [00:07<00:11,  3.77s/chunk]

Merging docs_mct_epr.csv: 100%|██████████| 5/5 [00:07<00:00,  1.16s/chunk]

Merging docs_mct_epr.csv: 100%|██████████| 5/5 [00:07<00:00,  1.58s/chunk]

### Build all annotation batches dataframe:

In [22]:
# This will merge all annotation source batches into a single file. This is useful for filtering. May produce a large file.

from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_annot_df

all_pat_list = pat2vec_obj.all_patient_list

dfa = build_merged_epr_mct_annot_df(
    all_pat_list, pat2vec_obj.config_obj, overwrite=True
)

dfa = pd.read_csv(dfa)

dfa

2026-08-05 20:45:34,800 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: annots_mct_epr.csv. Processing 5 patient(s) with chunk_size=500.


Merging annots_mct_epr.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

2026-08-05 20:45:35,579 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for annotations.ann_textual_obs: Table 'annotations_ann_textual_obs' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:35,581 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for annotations.ann_reports: Table 'annotations_ann_reports' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:35,581 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for annotations.ann_epic_medical_history: Table 'annotations_ann_epic_medical_history' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


...

2026-08-05 20:45:35,983 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for annotations.ann_epic_orders: Table 'annotations_ann_epic_orders' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


2026-08-05 20:45:35,984 - pat2vec.util.helper_functions - ERROR - Error in get_df_from_db reading from database for annotations.ann_epic_clinical_notes_appointments: Table 'annotations_ann_epic_clinical_notes_appointments' not found in database. This likely means dummy data was never generated during testing mode. Please ensure the pat2vec initialization successfully saved batch data.


Merging annots_mct_epr.csv: 100%|██████████| 1/1 [00:01<00:00,  1.18s/chunk]

Merging annots_mct_epr.csv: 100%|██████████| 1/1 [00:01<00:00,  1.18s/chunk]

,client_idcode,pretty_name,cui,type_ids,types,source_value,detected_name,acc,context_similarity,start,...,Time_Confidence,Presence_Value,Presence_Confidence,Subject_Value,Subject_Confidence,updatetime,annotation_batch_source,document_guid,annotation_description,observationannotation_recordeddtm
0,P0IFD0TV,"Hypertensive disorder, systemic arterial (diso...",38341003,"[""T-11""]","[""disorder""]",Hypertension,hypertension,0.943669,0.752522,207,...,0.999992,True,1.0,Patient,1.000000,1995-11-07 19:44:52.000000,epr,2b861723,NaN,NaN
1,P0IFD0TV,Diarrhea (finding),62315008,"[""T-18""]","[""finding""]",diarrhea,diarrhea,0.163894,0.667993,3,...,1.000000,True,1.0,Other,0.996056,1995-11-07 19:44:52.000000,epr,2b861723,NaN,NaN
2,P0IFD0TV,Ex-smoker (finding),8517006,"[""T-18""]","[""finding""]",former smoker,former~smoker,1.000000,1.000000,74,...,0.999999,True,1.0,Patient,0.999997,1995-11-07 19:44:52.000000,epr,2b861723,NaN,NaN
3,P0IFD0TV,Persistent cough (finding),284523002,"[""T-18""]","[""finding""]",persistent cough,persistent~cough,0.329012,0.102714,5,...,0.999999,True,1.0,Other,0.999982,1995-11-07 19:44:52.000000,epr,2b861723,NaN,NaN
4,P0IFD0TV,Community acquired pneumonia (disorder),385093006,"[""T-11""]","[""disorder""]",Community-acquired pneumonia,community~acquired~pneumonia,0.818049,0.406305,303,...,0.999999,False,1.0,Patient,1.000000,1995-11-07 19:44:52.000000,epr,2b861723,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,VF8MDD4V,"Hypertensive disorder, systemic arterial (diso...",38341003,"[""T-11""]","[""disorder""]",Hypertension,hypertension,0.943669,0.752522,207,...,0.999992,True,1.0,Patient,1.000000,2002-02-16 15:54:19.000000,epic_clinical_notes,7f39fe8e,NaN,NaN
166,VF8MDD4V,Diarrhea (finding),62315008,"[""T-18""]","[""finding""]",diarrhea,diarrhea,0.163894,0.667993,3,...,1.000000,True,1.0,Other,0.996056,2002-02-16 15:54:19.000000,epic_clinical_notes,7f39fe8e,NaN,NaN
167,VF8MDD4V,Ex-smoker (finding),8517006,"[""T-18""]","[""finding""]",former smoker,former~smoker,1.000000,1.000000,74,...,0.999999,True,1.0,Patient,0.999997,2002-02-16 15:54:19.000000,epic_clinical_notes,7f39fe8e,NaN,NaN
168,VF8MDD4V,Persistent cough (finding),284523002,"[""T-18""]","[""finding""]",persistent cough,persistent~cough,0.329012,0.102714,5,...,0.999999,True,1.0,Other,0.999982,2002-02-16 15:54:19.000000,epic_clinical_notes,7f39fe8e,NaN,NaN


### Build additional batches from individual patient data batches

In [23]:
# This will merge all drug source batches into a single file. This is useful for filtering. May produce a large file.

from pat2vec.util.post_processing_build_methods import merge_drugs_csv

all_pat_list = pat2vec_obj.all_patient_list

merged_drugs_path = merge_drugs_csv(
    all_pat_list, pat2vec_obj.config_obj, overwrite=True
)

merged_drugs = pd.read_csv(merged_drugs_path)
merged_drugs

2026-08-05 20:45:36,378 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: merged_drugs.csv. Processing 5 patient(s) with chunk_size=500.


Merging merged_drugs.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

Merging merged_drugs.csv: 100%|██████████| 1/1 [00:00<00:00,  2.64chunk/s]

,client_idcode,order_guid,order_name,order_summaryline,order_holdreasontext,order_entered,clientvisit_visitidcode,order_performeddtm,order_createdwhen
0,P0IFD0TV,ba81edd9-587e-4344-af3f-920c98b8e4cc,Omalizumab,Medical project for recent never. Inside wait ...,Treat seat strategy. Simply discover soon desp...,2002-06-22T06:46:33,visit_0,2019-06-05T22:16:19,1996-02-03T22:16:08
1,P0IFD0TV,9b3080d5-6fb7-4271-904d-281fc9535b63,Naproxen,Lead soon property write. Fish sense kind spri...,NaN,2018-04-18T02:09:15,visit_1,2005-09-15T16:20:05,2022-05-23T18:15:47
2,P0IFD0TV,629c2ae3-1d9a-4659-82ec-9f2dfbf6e16f,Terbutaline,Right subject try wonder move trade. North agr...,NaN,2006-01-13T01:05:35,visit_2,2002-03-27T10:20:11,2005-09-25T07:32:15
3,P0IFD0TV,41357e8c-30a9-40ad-939b-462de645f129,Venlafaxine,Sure outside building worker site. Mouth produ...,NaN,2023-09-05T14:22:33,visit_3,2017-04-20T12:11:49,2008-08-25T12:03:34
4,P0IFD0TV,006ed6e3-6fa1-4735-b572-f3d00b5cea6a,Ipratropium,Hard network gas you nearly goal law fill. Bui...,Begin most heavy. Game have return since nothi...,2021-02-21T06:26:59,visit_4,2011-12-27T05:31:24,2004-02-10T08:49:43
5,P0IFD0TV,89d7fd6c-ce77-4f00-acf2-7e7685197ff4,Theophylline,Difficult mission late kind team wrong figure ...,Face if whom commercial way least. Because suc...,2021-05-27T18:59:26,visit_5,2008-06-17T17:48:36,1997-04-02T13:31:12
6,P0IFD0TV,bdf070aa-f0b5-456b-b82c-9074afd5dea5,Hydrocodone,Prepare trouble consider one play man before. ...,Past medical leg never. Last special prepare. ...,1996-11-26T04:08:35,visit_6,2016-11-15T02:02:20,2002-03-07T00:04:59
7,P0IFD0TV,5d3d9e56-3270-44fa-abae-4f43bcae8081,Omalizumab,Everybody so increase various. Environment abl...,NaN,2002-10-17T12:14:54,visit_7,2010-08-12T09:53:46,2014-04-19T16:02:23
8,P0IFD0TV,aa0b7b14-f2e9-402d-91e9-cdaa6e6981a3,Flunisolide,Range explain dinner bed within set region bey...,Yes heart agreement us stuff practice. Case ex...,2022-12-18T00:10:18,visit_8,1999-11-11T13:06:40,2024-10-26T02:49:31
9,P64CIYHE,d6707a21-29dc-473e-9820-b3f25989dd6c,Salbutamol,Agree anyone take sister Democrat. Positive qu...,Interest heavy give edge level. Station howeve...,2002-03-07T00:04:59,visit_0,2005-09-15T16:20:05,2024-10-26T02:49:31


In [24]:
# dfmdi = pd.read_csv('new_project/merged_input_pat_batches/merged_drugs_batches.csv')

In [25]:
# for col in dfmdi.select_dtypes(exclude=[np.number]).columns:
#     assert dfmdi[col].astype(str).equals(merged_drugs[col].astype(str)), f"Mismatch in column: {col}"

In [26]:
# This will merge all diagnostics source batches into a single file. This is useful for filtering. May produce a large file.

from pat2vec.util.post_processing_build_methods import merge_diagnostics_csv

all_pat_list = pat2vec_obj.all_patient_list

merged_diagnostics_path = merge_diagnostics_csv(
    all_pat_list, pat2vec_obj.config_obj, overwrite=True
)

merged_diagnostics = pd.read_csv(merged_diagnostics_path)

2026-08-05 20:45:36,958 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: merged_diagnostics.csv. Processing 5 patient(s) with chunk_size=500.


Merging merged_diagnostics.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

Merging merged_diagnostics.csv: 100%|██████████| 1/1 [00:00<00:00,  2.50chunk/s]

In [27]:
from pat2vec.util.post_processing_build_methods import merge_news_csv

all_pat_list = pat2vec_obj.all_patient_list

merged_news_path = merge_news_csv(all_pat_list, pat2vec_obj.config_obj, overwrite=True)

# merged_news = pd.read_csv(merged_news_path)

2026-08-05 20:45:37,554 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: merged_news.csv. Processing 5 patient(s) with chunk_size=500.


Merging merged_news.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

Merging merged_news.csv: 100%|██████████| 1/1 [00:00<00:00,  2.54chunk/s]

In [28]:
from pat2vec.util.post_processing_build_methods import merge_bmi_csv

all_pat_list = pat2vec_obj.all_patient_list

merged_bmi_path = merge_bmi_csv(all_pat_list, pat2vec_obj.config_obj, overwrite=True)

# merged_bmi = pd.read_csv(merged_bmi_path)

2026-08-05 20:45:38,148 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: merged_bmi.csv. Processing 5 patient(s) with chunk_size=500.


Merging merged_bmi.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

Merging merged_bmi.csv: 100%|██████████| 1/1 [00:00<00:00,  2.66chunk/s]

In [29]:
from pat2vec.util.post_processing_build_methods import build_merged_bloods

all_pat_list = pat2vec_obj.all_patient_list

merged_bloods_path = build_merged_bloods(
    all_pat_list, pat2vec_obj.config_obj, overwrite=True
)

merged_bloods = pd.read_csv(merged_bloods_path)
merged_bloods

2026-08-05 20:45:38,725 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: bloods_batches.csv. Processing 5 patient(s) with chunk_size=500.


Merging bloods_batches.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

Merging bloods_batches.csv: 100%|██████████| 1/1 [00:00<00:00,  2.54chunk/s]

,client_idcode,basicobs_itemname_analysed,basicobs_value_numeric,basicobs_entered,clientvisit_serviceguid,updatetime
0,P0IFD0TV,Glucose,74.413498,2023-06-15T20:07:01,service_0,2023-06-15T20:07:01
1,P64CIYHE,Glucose,74.413498,2023-06-15T20:07:01,service_0,2023-06-15T20:07:01
2,V945NQ4F,Glucose,74.413498,2023-06-15T20:07:01,service_0,2023-06-15T20:07:01
3,VC58DRC1,Glucose,74.413498,2023-06-15T20:07:01,service_0,2023-06-15T20:07:01
4,VF8MDD4V,Glucose,74.413498,2023-06-15T20:07:01,service_0,2023-06-15T20:07:01


In [30]:
# pd.read_csv('new_project/merged_input_pat_batches/merged_bloods_batches.csv')

In [31]:
from pat2vec.util.post_processing_build_methods import merge_demographics_csv

all_pat_list = pat2vec_obj.all_patient_list

merged_demographics_path = merge_demographics_csv(
    all_pat_list, pat2vec_obj.config_obj, overwrite=True
)

merged_demographics = pd.read_csv(merged_demographics_path)

merged_demographics

2026-08-05 20:45:39,317 - pat2vec.util.post_processing_build_methods - INFO - GENERIC BUILDER START: merged_demographics.csv. Processing 5 patient(s) with chunk_size=500.


Merging merged_demographics.csv:   0%|          | 0/1 [00:00<?, ?chunk/s]

Merging merged_demographics.csv: 100%|██████████| 1/1 [00:00<00:00,  2.40chunk/s]

,client_idcode,client_firstname,client_lastname,client_dob,client_gendercode,client_racecode,client_deceaseddtm,updatetime
0,P0IFD0TV,Michael,Quinn,1981-06-15T00:00:00,female,NaN,NaN,2019-11-02
1,P0IFD0TV,NaN,Quinn,1981-06-15T00:00:00,NaN,Bantu,NaN,2018-06-22
2,P0IFD0TV,Michael,Quinn,1981-06-15T00:00:00,female,Bantu,NaN,2001-11-04
3,P0IFD0TV,Michael,Quinn,1981-06-15T00:00:00,NaN,NaN,NaN,2019-04-05
4,P0IFD0TV,Michael,Quinn,NaN,female,Bantu,NaN,2005-08-09
5,P0IFD0TV,Michael,Quinn,1981-06-15T00:00:00,female,NaN,NaN,2008-07-29
6,P0IFD0TV,Michael,NaN,1981-06-15T00:00:00,female,Bantu,NaN,2017-11-08
7,P0IFD0TV,NaN,Quinn,1981-06-15T00:00:00,female,Bantu,NaN,2017-02-10
8,P64CIYHE,Richard,Flores,NaN,male,NaN,NaN,2024-10-19
9,P64CIYHE,Richard,Flores,1945-10-22T00:00:00,male,Gabonese British,NaN,2019-06-27


### Quick Access Methods for Each Table Type

The `pat2vec_obj` provides dot notation methods for each table type:


In [32]:
# Get Epic patients data for a single patient
demo_data = pat2vec_obj.get_epic_patients(patient_id=pat2vec_obj.all_patient_list[0])
print(f"Epic patients: {demo_data.shape}")

Saving epic patient data to /workspaces/pat2vec/notebooks/new_project/new_project/epic_patients_results.csv
Epic patients: (5, 13)
